# Day 1 / Session 2 -- Prompt Engineering Demos

Six live demos that show how prompt design controls LLM behavior.
Run every cell top-to-bottom; each demo builds your intuition for
the patterns you will apply when you build your own agent components later.

## Session Goal

By the end of this session you will master **6 prompt engineering patterns** that control LLM behavior:

| # | Pattern | What It Does |
|---|---------|-------------|
| 1 | **Few-Shot Learning** | Provide examples so the model learns your expected format and behavior |
| 2 | **Chain-of-Thought** | Force step-by-step reasoning for math, logic, and multi-step problems |
| 3 | **Role-Based Personas** | Use system prompts to create specialized agent personalities |
| 4 | **JSON Structured Output** | Guarantee machine-readable responses for downstream pipelines |
| 5 | **Multi-Turn Conversation** | Maintain context across multiple exchanges for stateful agents |
| 6 | **LangChain Templates** | Compose reusable prompt + model + parser chains |

These are the **building blocks for creating reliable, production-grade AI agents**. Every agent you build on Day 2 and Day 3 will use one or more of these patterns under the hood. The demos progress from simple (controlling output format) to complex (composable chains), so pay attention to how each pattern solves a different reliability problem.

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn langchain langchain-openai langchain-core python-dotenv

## Environment Setup

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import json

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.
**Few-shot** prompting provides a handful of examples so the model can learn
the expected format and behavior.

We compare both approaches on a **client feedback classification** task --
categorizing engagement survey responses as positive, negative, or neutral.

### Demo 1a: Zero-Shot Classification

In [ ]:
# Demo 1a: Zero-shot -- no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

**Observe:** The zero-shot response likely includes extra explanation and may not follow a consistent format. This is fine for exploration but unreliable for production pipelines.

### Demo 1b: Few-Shot Classification

In [ ]:
# Demo 1b: Few-shot -- provide labeled examples from past engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations \u2014 the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Key Insight:** The few-shot response follows the exact format shown in the examples (single word: POSITIVE/NEGATIVE/NEUTRAL). Few-shot examples act as an implicit 'output schema' that the model learns to follow.

### Try This
Add a 4th few-shot example with an ambiguous feedback statement. Does the model classify it consistently if you run the cell multiple times?

---
## Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning
process step by step, which improves accuracy on math, logic, and multi-step
problems.

This mirrors structured problem-solving -- breaking a problem into steps
before jumping to the answer. CoT is especially valuable for market sizing
and financial analysis.

### Demo 2a: Without Chain-of-Thought

In [ ]:
# Demo 2a: Without CoT
problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

**Notice:** The model may jump straight to an answer. With complex math, this increases the chance of errors.

### Demo 2b: With Chain-of-Thought

In [ ]:
# Demo 2b: With explicit CoT -- mirroring structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Gotcha:** CoT works best for math, logic, and multi-step reasoning. For simple classification tasks, it can actually make responses worse by overthinking. Match the technique to the task!

### Quick Recap
- **Q:** When would you use zero-shot vs few-shot? Give a consulting scenario for each.
- **Q:** A colleague says 'CoT always makes LLM responses better.' Do you agree? Why or why not?

---
## Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different practice-area
personas that approach the same client question from different angles. This is
foundational for building specialized agents in multi-agent systems.

### Demo 3a: Define Personas and Question

In [ ]:
# Demo 3a: Define personas dict and the client question
personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

print(f"Question: {question}")
print(f"\nPersonas defined: {list(personas.keys())}")

### Demo 3b: Run All Personas

In [ ]:
# Demo 3b: Run the loop -- each persona responds to the same question
for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")

**Observation:** Each persona focuses on different aspects -- growth levers, regulatory risk, or talent. In Day 2, you'll wire these into separate agent nodes that collaborate.

### Try This
Change the user question to a different industry (e.g., 'Our airline client is losing market share to low-cost carriers'). How do the persona responses adapt?

---
## Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems, we often need the LLM output to be
machine-readable -- for example, extracting structured client data from
meeting notes or parsing engagement details from unstructured text.
JSON mode ensures the response is valid JSON, making downstream processing
reliable.

### Demo 4a: Make the API Call with JSON Mode

In [ ]:
# Demo 4a: Make the API call with JSON mode
text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

print("Raw JSON response:")
print(response.choices[0].message.content)

### Demo 4b: Parse and Access Fields

In [ ]:
# Demo 4b: Parse JSON and demonstrate programmatic field access
parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

**Gotcha:** `response_format={'type': 'json_object'}` guarantees valid JSON, but it does NOT guarantee the JSON matches your schema. Always validate the keys you expect! In Demo 6 Part B, Pydantic solves this.

### Try This
Change the text to a different executive bio. Does the JSON extraction still work correctly? What happens if a field is missing from the text?

### Quick Recap
- **Q:** You need to extract 10 fields from a client's earnings call transcript. Which combination of techniques from Demos 3 and 4 would you use?
- **Q:** What's the risk of using JSON mode without Pydantic validation?

---
## Demo 5: Multi-Turn Conversation Management

Agents need to maintain context across multiple exchanges -- remembering the
client industry, engagement scope, and prior recommendations. This demo shows
how to build a conversation manager that accumulates context, mirroring how
an engagement progresses through multiple discussions.

### Demo 5a: Define the ConversationManager Class

In [ ]:
# Demo 5a: Define the ConversationManager class
class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

print("ConversationManager class defined.")
print("Key methods: send(user_message) -> str, get_history_length() -> int")

### Demo 5b: Run the Multi-Turn Conversation

In [ ]:
# Demo 5b: Run a multi-turn engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

**Key Insight:** The conversation history grows with every turn. In production, this means token costs increase over time. The Optional Exercise 3 in Session 1 explores strategies to handle this.

---
## Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its
`ChatPromptTemplate` supports parameterized prompts, and `OutputParser`
classes enforce structured output formats (JSON, lists, etc.) -- reducing
boilerplate and improving reliability.

### Demo 6 Part A: ChatPromptTemplate with Variables

In [ ]:
# Demo 6 Part A: ChatPromptTemplate with variables
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

print("PART A: LangChain ChatPromptTemplate for Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print(f"Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print(f"Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

### Demo 6 Part B: Pydantic Structured Output

In [ ]:
# Demo 6 Part B: Pydantic-based Structured Output for deliverables
print("PART B: Pydantic Structured Output -- Engagement Summary")
print("=" * 60)

class EngagementSummary(BaseModel):
    executive_summary: str = Field(description="A 1-2 sentence executive summary suitable for a CEO briefing")
    key_findings: list[str] = Field(description="A list of 3 key findings")
    strategic_priority: str = Field(description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")

parser = JsonOutputParser(pydantic_object=EngagementSummary)
format_instructions = parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = parser.parse(response.content)

print("Parsed structured output:")
for key, value in parsed_output.items():
    print(f"  {key}: {value}")

### Demo 6 Part C: LangChain Chain (Prompt | LLM | Parser)

In [ ]:
# Demo 6 Part C: LangChain Chain -- Prompt | LLM | Parser
print("PART C: LangChain Chain -- Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(json.dumps(result, indent=2))

**Why This Matters:** The chain pattern `prompt | llm | parser` is the core abstraction in LangChain. Every agent you build on Day 2 uses this under the hood.

### Quick Recap
- **Q:** What happens to the conversation if you DON'T append the assistant's response back to the message history?
- **Q:** How does LangChain's `ChatPromptTemplate` differ from manually building the messages list?

---
## Key Takeaways

| Demo | Pattern | Why It Matters | Key Observation |
| --- | --- | --- | --- |
| 1 | Zero-shot vs. Few-shot | Few-shot examples lock down output format | Few-shot examples act as an implicit 'output schema' -- reliable for production pipelines |
| 2 | Chain-of-Thought | Step-by-step reasoning improves accuracy on quantitative tasks | CoT works best for math/logic; for simple classification it can overthink -- match technique to task |
| 3 | Role-based personas | System prompts define agent specialization | Each persona surfaces different insights; these become separate agent nodes on Day 2 |
| 4 | JSON mode | Machine-readable output for downstream pipelines | Guarantees valid JSON but NOT schema compliance -- always validate keys or use Pydantic |
| 5 | Multi-turn management | Context accumulation enables stateful agents | History grows every turn; token costs increase -- plan for context window management |
| 6 | LangChain templates | Reusable, composable prompt + parser chains | The `prompt \| llm \| parser` chain is the core abstraction behind every Day 2 agent |

Next up: **Exercises** -- you will build a research agent system prompt, a few-shot classifier, a CoT market sizing function, and more.